# Simple Database Table Wipe & Load Tool

This notebook provides a straightforward, step-by-step approach to:
1. Load a CSV file and preview the data
2. Wipe all existing data from either `apg_catalog` or `apg_content` table
3. Load the CSV data into the cleared table

**⚠️ WARNING**: This tool will DELETE ALL DATA in the selected table before loading new data. Use with caution!

## Instructions
1. Update the configuration in the first cell
2. Run each cell in order
3. Follow the prompts in each section

## 1. Configuration

**Important:** Update the variables below with your settings

In [ ]:
# --- Database Connection Parameters --- 
# !!! MODIFY THESE VALUES TO MATCH YOUR DATABASE SETUP !!!
DB_PARAMS = {
    "host": "localhost",      # e.g., 'localhost' or an IP address
    "port": "5432",           # Default PostgreSQL port
    "dbname": "maven-finance",  # Your database name
    "user": "iris_dev",       # Your database username
    "password": ""             # Your database password (leave empty if none)
}

# --- Target Table Configuration ---
# Choose which table to wipe and load: either "apg_catalog" or "apg_content"
TARGET_TABLE = "apg_catalog"  # <-- CHANGE THIS TO YOUR TARGET TABLE

# --- CSV File Path ---
# Provide the full path to your deployment CSV file
CSV_FILE_PATH = "catalog_2024-01-15_10-30-00.csv"  # <-- CHANGE THIS TO YOUR CSV FILE PATH

print(f"Configuration set:")
print(f"  Database: {DB_PARAMS['dbname']}@{DB_PARAMS['host']}")
print(f"  Target table: {TARGET_TABLE}")
print(f"  CSV file: {CSV_FILE_PATH}")

## 2. Setup and Imports

In [ ]:
import pandas as pd
import psycopg2
import psycopg2.extras
from psycopg2 import sql
import os
import logging
import json
from datetime import datetime
import io
from typing import Optional, Dict, Any, Tuple

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Register UUID adapter
try:
    psycopg2.extras.register_uuid()
except Exception as e:
    logger.warning(f"Could not register UUID adapter: {e}")

# Validate configuration
SUPPORTED_TABLES = ["apg_catalog", "apg_content"]
if TARGET_TABLE not in SUPPORTED_TABLES:
    raise ValueError(f"Invalid TARGET_TABLE: {TARGET_TABLE}. Must be one of {SUPPORTED_TABLES}")

print("✅ Setup complete")

## 3. Helper Functions

In [ ]:
def connect_to_db(db_params: Dict[str, str]) -> Optional[psycopg2.extensions.connection]:
    """
    Connects to the PostgreSQL database using the provided parameters.
    """
    try:
        conn = psycopg2.connect(**db_params)
        conn.autocommit = False
        return conn
    except Exception as e:
        logger.error(f"Error connecting to database: {e}")
        return None


def get_table_count(conn: psycopg2.extensions.connection, table_name: str) -> int:
    """
    Get the current row count for a table.
    """
    try:
        with conn.cursor() as cur:
            query = sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(table_name))
            cur.execute(query)
            return cur.fetchone()[0]
    except Exception as e:
        logger.error(f"Error getting count for table {table_name}: {e}")
        return -1


def validate_csv_for_table(df: pd.DataFrame, table_name: str) -> Tuple[bool, list]:
    """
    Validate that the CSV data matches the expected schema for the table.
    """
    issues = []
    
    # Define required columns for each table (excluding auto-generated fields)
    required_columns = {
        "apg_catalog": [
            "document_source", "document_type", "document_name"
        ],
        "apg_content": [
            "document_source", "document_type", "document_name",
            "section_id", "section_content"
        ]
    }
    
    # Check for required columns
    if table_name in required_columns:
        missing_cols = [col for col in required_columns[table_name] if col not in df.columns]
        if missing_cols:
            issues.append(f"Missing required columns: {', '.join(missing_cols)}")
    
    # Check for empty dataframe
    if df.empty:
        issues.append("CSV file is empty")
    
    # Check for auto-generated fields that should not be in the CSV
    auto_fields = ["id", "created_at"]
    present_auto_fields = [field for field in auto_fields if field in df.columns]
    if present_auto_fields:
        issues.append(f"CSV contains auto-generated fields that will be ignored: {', '.join(present_auto_fields)}")
    
    return (len(issues) == 0, issues)


def preprocess_dataframe(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """
    Preprocess the DataFrame to match PostgreSQL requirements.
    """
    df_processed = df.copy()
    
    # Remove any auto-generated fields if present
    auto_fields = ["id", "created_at"]
    for field in auto_fields:
        if field in df_processed.columns:
            df_processed = df_processed.drop(field, axis=1)
    
    # Handle numeric fields
    numeric_fields = {
        "apg_catalog": ["file_size"],
        "apg_content": ["section_id", "page_number"]
    }
    
    if table_name in numeric_fields:
        for field in numeric_fields[table_name]:
            if field in df_processed.columns:
                df_processed[field] = df_processed[field].replace(['', 'NULL'], pd.NA)
                df_processed[field] = pd.to_numeric(df_processed[field], errors='coerce')
    
    # Handle timestamp fields
    timestamp_fields = ["date_created", "date_last_modified"]
    for field in timestamp_fields:
        if field in df_processed.columns:
            df_processed[field] = df_processed[field].replace('NULL', pd.NA)
            df_processed[field] = pd.to_datetime(df_processed[field], errors='coerce')
    
    # Handle embedding fields (should be JSON strings or NULL)
    embedding_fields = ["document_usage_embedding", "document_description_embedding"]
    for field in embedding_fields:
        if field in df_processed.columns:
            df_processed[field] = df_processed[field].replace('NULL', None)
            df_processed[field] = df_processed[field].replace('', None)
    
    # Clean text fields
    text_fields = ["document_description", "document_usage", "section_summary", "section_content",
                   "section_name", "document_source", "document_type", "document_name",
                   "file_name", "file_type", "file_path", "file_link"]
    for field in text_fields:
        if field in df_processed.columns:
            df_processed[field] = df_processed[field].replace('NULL', None)
            df_processed[field] = df_processed[field].astype(str).str.replace('\x00', '', regex=False)
            df_processed[field] = df_processed[field].replace('nan', None)
    
    return df_processed

print("✅ Helper functions loaded")

## 4. Load and Validate CSV File

This section loads your CSV file and validates it against the target table schema.

In [ ]:
# Load CSV file
print(f"Loading CSV file: {CSV_FILE_PATH}")
print("=" * 80)

try:
    # Check if file exists
    if not os.path.exists(CSV_FILE_PATH):
        raise FileNotFoundError(f"CSV file not found: {CSV_FILE_PATH}")
    
    # Load the CSV
    df = pd.read_csv(CSV_FILE_PATH)
    print(f"✅ Successfully loaded CSV file")
    print(f"   - Rows: {len(df):,}")
    print(f"   - Columns: {len(df.columns)}")
    
    # Validate the CSV
    print(f"\nValidating CSV for table '{TARGET_TABLE}'...")
    is_valid, validation_issues = validate_csv_for_table(df, TARGET_TABLE)
    
    if validation_issues:
        print("\nValidation issues found:")
        for issue in validation_issues:
            if "auto-generated fields" in issue:
                print(f"   ⚠️  {issue}")
            else:
                print(f"   ❌ {issue}")
    
    # Check if we can proceed
    critical_issues = [i for i in validation_issues if "auto-generated fields" not in i]
    if critical_issues:
        print("\n❌ CRITICAL VALIDATION ERRORS - Cannot proceed!")
        print("Please fix the CSV file and try again.")
    else:
        print("\n✅ CSV validation passed - ready to proceed")
        
        # Show preview
        print("\nData Preview (first 5 rows):")
        print("=" * 80)
        display(df.head())
        
        # Show column information
        print("\nColumn Information:")
        print("=" * 80)
        for col in df.columns:
            non_null = df[col].notna().sum()
            null_pct = ((len(df) - non_null) / len(df) * 100) if len(df) > 0 else 0
            print(f"  {col:<30} {df[col].dtype:<15} {non_null:>8,}/{len(df):,} non-null ({null_pct:>5.1f}% null)")
        
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("\nPlease check your CSV_FILE_PATH in the configuration cell and try again.")

## 5. Test Database Connection

Let's verify the database connection and check the current state of the target table.

In [ ]:
print("Testing database connection...")
print("=" * 80)

conn = None
try:
    # Connect to database
    conn = connect_to_db(DB_PARAMS)
    if conn:
        print(f"✅ Successfully connected to database: {DB_PARAMS['dbname']}")
        
        # Get current record count
        current_count = get_table_count(conn, TARGET_TABLE)
        if current_count >= 0:
            print(f"\nCurrent state of table '{TARGET_TABLE}':")
            print(f"  - Record count: {current_count:,}")
            
            # Show a sample of existing data if any
            if current_count > 0:
                with conn.cursor() as cur:
                    cur.execute(sql.SQL("SELECT * FROM {} LIMIT 3").format(sql.Identifier(TARGET_TABLE)))
                    columns = [desc[0] for desc in cur.description]
                    rows = cur.fetchall()
                    sample_df = pd.DataFrame(rows, columns=columns)
                    print("\nSample of existing data (first 3 rows):")
                    display(sample_df)
        else:
            print(f"\n⚠️ Could not get record count for table '{TARGET_TABLE}'")
            
        # Close connection for now
        conn.close()
        print("\n✅ Database connection test successful")
    else:
        print("\n❌ Failed to connect to database")
        print("Please check your DB_PARAMS configuration and try again.")
        
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    if conn and not conn.closed:
        conn.close()

## 6. WIPE TABLE - DANGER ZONE

**⚠️ WARNING**: The next cell will DELETE ALL DATA from the target table!

To proceed with wiping the table:
1. Change `CONFIRM_WIPE = False` to `CONFIRM_WIPE = True` in the cell below
2. Run the cell

This action cannot be undone!

In [ ]:
# SAFETY: Change this to True to confirm table wipe
CONFIRM_WIPE = False  # <-- CHANGE TO True TO PROCEED WITH WIPE

if not CONFIRM_WIPE:
    print("❌ Table wipe NOT executed - CONFIRM_WIPE is False")
    print("\nTo wipe the table:")
    print("1. Change CONFIRM_WIPE = False to CONFIRM_WIPE = True")
    print("2. Run this cell again")
else:
    print(f"🚨 WIPING TABLE '{TARGET_TABLE}'...")
    print("=" * 80)
    
    conn = None
    try:
        # Connect to database
        conn = connect_to_db(DB_PARAMS)
        if not conn:
            raise Exception("Failed to connect to database")
            
        # Get count before deletion
        count_before = get_table_count(conn, TARGET_TABLE)
        print(f"Records before wipe: {count_before:,}")
        
        # Delete all records
        with conn.cursor() as cur:
            delete_query = sql.SQL("DELETE FROM {}").format(sql.Identifier(TARGET_TABLE))
            cur.execute(delete_query)
            deleted_count = cur.rowcount
            
        # Commit the transaction
        conn.commit()
        
        # Get count after deletion
        count_after = get_table_count(conn, TARGET_TABLE)
        
        print(f"\n✅ Table wipe completed:")
        print(f"   - Records deleted: {deleted_count:,}")
        print(f"   - Records remaining: {count_after:,}")
        
        if count_after == 0:
            print(f"\n✅ Table '{TARGET_TABLE}' is now empty and ready for new data")
        else:
            print(f"\n⚠️ WARNING: Table still contains {count_after} records")
            
        conn.close()
        
    except Exception as e:
        print(f"\n❌ ERROR during table wipe: {e}")
        if conn and not conn.closed:
            conn.rollback()
            conn.close()
        print("\nNo changes were made to the database.")

## 7. Load Data into Table

Now we'll load the CSV data into the empty table.

To proceed:
1. Make sure you've successfully wiped the table in the previous step
2. Change `CONFIRM_LOAD = False` to `CONFIRM_LOAD = True`
3. Run the cell

In [ ]:
# SAFETY: Change this to True to confirm data load
CONFIRM_LOAD = False  # <-- CHANGE TO True TO PROCEED WITH LOAD

if not CONFIRM_LOAD:
    print("❌ Data load NOT executed - CONFIRM_LOAD is False")
    print("\nTo load the data:")
    print("1. Change CONFIRM_LOAD = False to CONFIRM_LOAD = True")
    print("2. Run this cell again")
else:
    print(f"📤 Loading data into table '{TARGET_TABLE}'...")
    print("=" * 80)
    
    conn = None
    try:
        # Preprocess the dataframe
        print("Preprocessing data...")
        df_processed = preprocess_dataframe(df, TARGET_TABLE)
        print(f"  - Original columns: {len(df.columns)}")
        print(f"  - Processed columns: {len(df_processed.columns)}")
        print(f"  - Rows to load: {len(df_processed):,}")
        
        # Connect to database
        conn = connect_to_db(DB_PARAMS)
        if not conn:
            raise Exception("Failed to connect to database")
            
        # Check table is empty (safety check)
        current_count = get_table_count(conn, TARGET_TABLE)
        if current_count > 0:
            print(f"\n⚠️ WARNING: Table '{TARGET_TABLE}' is not empty ({current_count:,} records)")
            print("Consider wiping the table first to avoid duplicate data.")
            
        # Load data using COPY
        print("\nLoading data using PostgreSQL COPY...")
        
        # Create CSV buffer
        csv_buffer = io.StringIO()
        df_processed.to_csv(csv_buffer, index=False, header=True, na_rep='\\N')
        csv_buffer.seek(0)
        
        with conn.cursor() as cur:
            # Build COPY query
            columns = df_processed.columns.tolist()
            copy_query = sql.SQL("COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE, NULL '\\N')").format(
                sql.Identifier(TARGET_TABLE),
                sql.SQL(', ').join(map(sql.Identifier, columns))
            )
            
            # Execute COPY
            cur.copy_expert(copy_query, csv_buffer)
            loaded_count = cur.rowcount
            
        # Commit the transaction
        conn.commit()
        
        # Verify final count
        final_count = get_table_count(conn, TARGET_TABLE)
        
        print(f"\n✅ Data load completed:")
        print(f"   - Records loaded: {len(df_processed):,}")
        print(f"   - Table now contains: {final_count:,} records")
        
        # Show sample of loaded data
        print("\nSample of loaded data (first 3 rows):")
        with conn.cursor() as cur:
            cur.execute(sql.SQL("SELECT * FROM {} LIMIT 3").format(sql.Identifier(TARGET_TABLE)))
            columns = [desc[0] for desc in cur.description]
            rows = cur.fetchall()
            sample_df = pd.DataFrame(rows, columns=columns)
            display(sample_df)
            
        conn.close()
        print("\n✅ SUCCESS: Data has been loaded into the database!")
        
    except Exception as e:
        print(f"\n❌ ERROR during data load: {e}")
        if conn and not conn.closed:
            conn.rollback()
            conn.close()
        print("\nNo data was loaded to the database.")

## 8. Summary

Run this cell to see a final summary of the operation.

In [ ]:
print("Operation Summary")
print("=" * 80)

conn = None
try:
    conn = connect_to_db(DB_PARAMS)
    if conn:
        # Get final counts
        final_count = get_table_count(conn, TARGET_TABLE)
        
        print(f"Target table: {TARGET_TABLE}")
        print(f"CSV file: {CSV_FILE_PATH}")
        print(f"CSV rows: {len(df):,}")
        print(f"Final table count: {final_count:,}")
        
        # Get document source breakdown if applicable
        if 'document_source' in df.columns:
            print("\nDocument sources in loaded data:")
            source_counts = df['document_source'].value_counts()
            for source, count in source_counts.items():
                print(f"  - {source}: {count:,} records")
                
        conn.close()
    else:
        print("Could not connect to database for summary")
        
except Exception as e:
    print(f"Error generating summary: {e}")
    if conn and not conn.closed:
        conn.close()

print("\n" + "=" * 80)
print("Process complete!")